# Lab 01A: Colab Setup & Good Prompting (no API key)

**Week 1 — Agentic AI: Building Autonomous Intelligent Systems**

This is the zero-setup warm-up lab. It uses Colab's **built-in AI** (`from google.colab import ai`), so there is **no API key to create, no secret to configure, and nothing to install** — Colab talks to Gemini for you. That makes it the fastest possible on-ramp to a working LLM call.

Once you can make a call, the lab spends its time on the one skill everything else in this course is built on: **writing good prompts**.

By the end you will be able to:

1. Make your first LLM call with Colab's built-in `ai` module — no key required
2. See a prompt for what it really is — a **structured set of System / User / Assistant messages** (and how you fold that structure into the prompt when the SDK is minimal)
3. Apply the **five core prompting principles** and techniques like few-shot and chain-of-thought
4. Read a prompt's **anatomy** (Role · Task · Constraints · Output Format …) and turn a messy request into a clean, structured answer

> **Which SDK?** This lab uses `google.colab.ai`, which is the simplest way to get started but is deliberately limited. The rest of the course (Lab 01A's sibling and Labs 02+) uses the full **`google-genai`** SDK with an API key, which adds dedicated system instructions, *guaranteed* structured output, tool use, and async. We'll flag those differences as they come up.

## Setup: nothing to set up

Colab ships with an `ai` module that is already authenticated for you. You just import it and call it. First, let's see which models are available — model names look like `google/...`.

In [ ]:
from google.colab import ai

# See what you can call.
for name in ai.list_models():
    print(name)

Pick one model and store it in a `MODEL` constant so the rest of the notebook is a one-line change. Then run a quick connectivity check.

> If the next cell raises a *model not found* error, copy one of the names printed above into `MODEL`.

In [ ]:
MODEL = "google/gemini-2.5-flash-lite"

# Quick connectivity check
response = ai.generate_text("Say 'Setup complete!' and nothing else.", model_name=MODEL)
print(response)

If you saw `Setup complete!`, you're connected to Gemini — no key, no install. 🎉

Notice the call is dead simple: `ai.generate_text(prompt, model_name=MODEL)` returns the text directly as a string. We wrap it in a one-line `generate()` helper so the rest of the notebook reads cleanly.

In [ ]:
def generate(prompt: str) -> str:
    """Send a single prompt to Gemini via Colab's built-in AI and return the text."""
    return ai.generate_text(prompt, model_name=MODEL).strip()


print(generate("In two sentences, explain what a large language model is to a curious beginner."))

Each call is **stateless** — the model has no memory of previous calls. If you want it to "remember" something, you have to include it in the prompt. Try running the cell again, or change the prompt and re-run, to feel how the output shifts.

### Bonus: streaming

Pass `stream=True` to get the answer token-by-token as it's generated — nice for long responses so the user isn't staring at a blank screen.

In [ ]:
for chunk in ai.generate_text("List 3 fun facts about octopuses.", model_name=MODEL, stream=True):
    print(chunk, end="")

---

## Part A — A prompt is a structured set of messages

It is tempting to think of a prompt as "the text you type." Conceptually it is really a **structured set of messages**, each with a **role**:

| Role | Who | What it does |
| :--- | :--- | :--- |
| ⚙️ **System** | you (the developer) | sets behavior — tone, rules, persona |
| 👤 **User** | the person | supplies the question, goal, and context |
| 💬 **Assistant** | the model | its replies; prior ones become context that steers the next turn |

> **Key idea:** a prompt is not just one string of text — it is a structured set of messages that steers the model's behavior.

> **SDK note:** The full `google-genai` SDK gives each role a dedicated slot (a `system_instruction` config and a list of role-tagged messages). The minimal `colab.ai` module only takes **one string**, so *you* recreate that structure inside the prompt text. Seeing it done by hand here makes the dedicated slots in the next labs click.

### System instructions: setting the model's role

A **system instruction** tells the model *how* to behave — its persona, tone, and rules — separately from the user's actual question. With this SDK we fold it into the prompt as a labeled section. Watch how the same question gets a very different answer depending on the system instruction.

In [ ]:
def generate_with_system(user_msg: str, system: str) -> str:
    """Compose a system + user 'message structure' into a single prompt."""
    prompt = f"""[System instructions]
{system}

[User]
{user_msg}"""
    return generate(prompt)


question = "How do I make a cup of tea?"

print("--- As a terse robot ---")
print(generate_with_system(question, "You are a terse robot. Answer in at most 3 short numbered steps."))

print("\n--- As an enthusiastic poet ---")
print(generate_with_system(question, "You are an enthusiastic poet. Answer as a short, joyful rhyming poem."))

---

## Part B — Core prompting principles

What you put *inside* those messages decides the quality of what comes back. Five principles carry most of the weight:

| Principle | In practice |
| :--- | :--- |
| **Clarity & specificity** | unambiguous task, format, and limits |
| **Conciseness** | if it's confusing to you, it will confuse the model |
| **Use action verbs** | *Summarize · Classify · Extract* — tell it the operation |
| **Instructions over constraints** | say what *to do*, not only what to avoid |
| **Experiment & iterate** | draft → test → refine → document |

> 📖 These come from **Appendix A: Advanced Prompting Techniques** (in Canvas). The cells below make them concrete — run each and compare the outputs; the difference *is* the lesson.

### Clarity & specificity (weak vs. strong)

A vague prompt makes the model guess what you want. Spell out the audience, length, format, and purpose. We'll pull a chunk of a public-domain book to summarize.

In [ ]:
import requests

URL = "https://www.gutenberg.org/files/11/11-0.txt"
TEXT = requests.get(URL).text[:6000]

print("--- First 1000 letters ---")
print(TEXT[:1000])

print("\n--- Weak prompt ---")
print(generate(f"""
create a study guide:

{TEXT}
"""))

print("\n--- Strong prompt ---")
print(generate(f"""
Create a beginner-friendly study guide from the text below.

Audience:
- First-year college students

Output format:
1. One-sentence overview
2. Five key events in order
3. Three vocabulary words with simple definitions
4. Two discussion questions
5. One common misunderstanding students might have

Rules:
- Use simple language.
- Do not assume students already know the story.
- Keep each section concise.

Text:
{TEXT}
"""))

### Instructions over constraints

A pile of "don't do this, don't do that" leaves the model guessing what it *should* do. Lead with a positive instruction; add a constraint only when it genuinely matters.

In [ ]:
topic = "how vaccines work"

print("--- All constraints (what to avoid) ---")
print(generate(f"Explain {topic}. Don't be too technical. Don't be too long. Don't use jargon."))

print("\n--- A positive instruction (what to do) ---")
print(generate(f"Explain {topic} in 2 plain-English sentences a 12-year-old would understand."))

### Applied techniques

Three reliable techniques that put the principles to work. **Use action verbs** (*Classify*, *Summarize*) and **iterate** on what you see.

**1. Show examples (zero-shot → few-shot).** Instead of *describing* the format you want, **demonstrate** it with a few worked examples. Zero examples is *zero-shot*, one is *one-shot*, several is *few-shot* — one of the most reliable ways to pin down tone, labels, and output shape.

In [ ]:
few_shot = generate(
    "Classify the sentiment of each product review as positive, negative, or neutral.\n\n"
    "Review: 'Absolutely loved it — best purchase I made all year!'\n"
    "Sentiment: positive\n\n"
    "Review: 'It broke after two days. Total waste of money.'\n"
    "Sentiment: negative\n\n"
    "Review: 'It arrived on time and works as described.'\n"
    "Sentiment: neutral\n\n"
    "Review: 'The colors are gorgeous, but the zipper already feels flimsy.'\n"
    "Sentiment:"
)
print(few_shot)

**2. Ask the model to reason step by step.** For anything involving logic, math, or multiple steps, asking the model to "think step by step" before answering often improves accuracy — it gives the model room to work through the problem instead of blurting out a guess. This is called **chain-of-thought** prompting.

In [ ]:
puzzle = '''
Write down the names of the first 5 months of the year in order. However, if a month's name contains the letter 'M', you must skip it entirely and replace it with the word 'SKIP'. Do not skip any other months.
'''

print("--- Just the answer ---")
print(generate(puzzle + " Reply with answer."))

print("\n--- Step by step ---")
print(generate(puzzle + '''
Think about this step-by-step before answering:
'''))

**3. Structure the prompt with Markdown.** When a prompt has several parts — a role, a task, rules, and the input — use **Markdown headings** to separate them. The model reads the structure clearly, and so do you. This is why Markdown is the lingua franca for prompts, README files, and agent instructions throughout this course.

In [ ]:
structured_prompt = """
# Role
You are an automated API Gateway and Data Sanitizer for a secure database pipeline.

# Task
Your core mission is to format raw corporate financial transactions into clean JSON objects. However, you must execute the Input Gate Validation rule first.

# Rules & Constraints
1. **Input Gate Validation (CRITICAL)**:
  Check the raw draft input.
  If the text mentions any financial transaction involving "cash",
  or if the date contains "2023", you must trigger the gate restriction.
  Immediately output exactly: `{"STATUS": "REJECTED_BY_GATE"}` and terminate.
  Do not process anything else.
2. If the gate check passes safely, extract the text into a clean JSON structure containing keys: `merchant`, `amount`, and `date`.
3. Output strictly valid JSON. Do not include markdown code block syntax (like ```json) or any conversational text.

# Raw Draft Input
Transaction clear date 11/12/2023. Paid merchant Acme Corp total amount forty dollars using a bank electronic wire transfer.
"""

print(generate(structured_prompt))

---

## Part C — Putting it together: prompt anatomy & structured output

Real prompts combine everything above into a System message (stable behavior) and a User message (the specific request), each with a recognizable **anatomy**: Role · Task · Context · Inputs · Constraints · Output Format · Eval Criteria.

Often you don't want a paragraph — you want **data** your code can use. With this minimal SDK, the only lever you have is to **ask for JSON in the prompt** and parse it yourself.

In [ ]:
import json


def generate_json(prompt: str):
    """Ask for JSON in the prompt, strip any code fences, and parse it."""
    raw = generate(prompt + "\n\nReturn ONLY valid JSON — no markdown fences, no prose.")
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)


notes = """
Meeting notes:
date = 2025-05-11

- Ana will update the Canvas instructions by Friday.
- We still need someone to test the Gemini setup in Colab.
- Joon said the Week 2 lab is light, so add one stronger structured-output exercise.
- Do not change the syllabus yet.
- The API key issue is blocking local setup for some students.
"""

prompt = f"""
Extract action items from the meeting notes as a JSON list of objects.
Each object has keys: task (string), owner (string or null), due_date (YYYY-MM-DD or null), priority ("low"|"medium"|"high"), blocked (boolean).

Rules:
- Only include real action items, not decisions or background.
- Use null if owner or due date is unknown.
- Mark blocked true only when the item is explicitly blocked.

Meeting notes:
{notes}
"""

for item in generate_json(prompt):
    print(item)

> ⚠️ **This works most of the time — and that's the problem.** Prompting for JSON can fail: the model may wrap it in ```` ```json ```` fences, add a sentence of explanation, or return an invalid label — and then `json.loads` throws. We patched the most common fence issue above, but it's a band-aid.
>
> This is exactly why the rest of the course uses the full **`google-genai`** SDK, which lets you hand the model a **schema** so the output is *guaranteed* to be valid, typed data — no parsing, no regex, no fallbacks. Getting bitten here is the motivation for the upgrade.

## A supporting concept: tokens and the context window

Models don't read characters or words — they read **tokens** (chunks of text, often a word or part of a word). The **context window** is the fixed budget of tokens available for everything in a single call: your instructions, the conversation transcript, and the model's reply all compete for the same space.

A long single "word" can be several tokens, while short common words are often one. This is why prompt length and cost are measured in tokens, not characters — and why **conciseness** is a prompting principle, not just good style. The longer your messages (and the longer the transcript you resend each turn), the more of the context window every call consumes.

> The full `google-genai` SDK exposes `client.models.count_tokens(...)` so you can measure this exactly; the minimal `colab.ai` module doesn't, so we keep it conceptual here.

## Your turn (exercises)

1. **Weak → strong.** Take a vague prompt of your own and rewrite it with a specific audience, length, and format. Run both and compare.
2. **Constraints → instructions.** Find a prompt you'd naturally phrase as a list of "don'ts" and rewrite it as a single positive instruction. Did the answer improve?
3. **Explore other "Chain-of-X" methods.** Design a prompt employing one of these strategies to solve a problem.

When you're done, save a copy (**File → Save a copy in Drive**) and submit your notebook link via Canvas.